## Bulding RAG with Lanchain and FAISS vector store

In [16]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

True

In [17]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20,
    length_function=len,
    separators=[" "]
)
chunks = text_splitter.split_documents(sample_documents)

print(f"Length if chunks: {len(chunks)}")
print(chunks[0])
print(chunks[1])

Length if chunks: 8
page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 381.36it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [20]:
query = "What is Ai?"
query_embedding = embeddings.embed_query(query)
query_embedding

[-0.02496449463069439,
 -0.009133679792284966,
 -0.0074615478515625,
 0.01500907726585865,
 0.013310405425727367,
 -0.010036046616733074,
 0.07456012070178986,
 0.04267145320773125,
 0.01698853075504303,
 0.05595095083117485,
 -0.029681894928216934,
 -0.004356078337877989,
 0.020532382652163506,
 -0.0482826791703701,
 -0.05866949260234833,
 0.04236198216676712,
 -0.0189194455742836,
 -0.052992478013038635,
 -0.0871717631816864,
 -0.0699818953871727,
 -0.00872023869305849,
 0.0196782685816288,
 -0.048600371927022934,
 -0.04870656877756119,
 -0.03246694430708885,
 0.09295571595430374,
 0.004014074802398682,
 -0.06721635162830353,
 -0.0021876483224332333,
 -0.011162949725985527,
 0.01210772804915905,
 -0.023686660453677177,
 0.10230045020580292,
 0.019198473542928696,
 -0.06650935113430023,
 0.04128839448094368,
 -0.040980417281389236,
 -0.027531692758202553,
 0.06851057708263397,
 -0.02964448556303978,
 -0.019207922741770744,
 -0.054202694445848465,
 0.01685267500579357,
 -0.071966238319

In [21]:
vectorstore = FAISS.from_documents(
    documents= chunks,
    embedding= embeddings
)

print(f"Number of Vectors in the store: {vectorstore.index.ntotal}")

Number of Vectors in the store: 8


In [22]:
vectorstore.save_local("faiss_index")
print("FAISS index saved locally as 'faiss_index' directory.")

FAISS index saved locally as 'faiss_index' directory.


In [23]:
vector_store_loaded = FAISS.load_local(
    "faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

In [24]:
query = "What is AI?"
res = vectorstore.similarity_search(query, k=2)
res

[Document(id='b8eaa799-63f7-4e61-88f8-398cf85f7e2b', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be'),
 Document(id='1b43fb9d-2583-4f95-853e-5dd572f82c0d', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='AI can be categorized into narrow AI and general AI.')]

In [25]:
query = "What is AI?"
res = vector_store_loaded.similarity_search(query, k=2)
res

[Document(id='b8eaa799-63f7-4e61-88f8-398cf85f7e2b', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be'),
 Document(id='1b43fb9d-2583-4f95-853e-5dd572f82c0d', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='AI can be categorized into narrow AI and general AI.')]

## Build RAG chain with LCEL

In [11]:
from langchain_groq import ChatGroq
import os

os.environ["GROQ_API_KEY"] = "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"

llm=ChatGroq(model="llama-3.1-8b-instant")


llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D5D1780290>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D5D1783450>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [14]:
llm.invoke("What the machine learning?")

AIMessage(content="Machine learning is a subset of artificial intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions based on that data. It's a field that has gained significant attention in recent years due to its potential to solve complex problems in various domains.\n\n**Key Concepts:**\n\n1. **Data**: Machine learning relies on large amounts of data to learn and improve its performance. This data can be in the form of text, images, audio, or any other type of data that can be represented digitally.\n2. **Algorithms**: Machine learning algorithms are the core of machine learning. These algorithms are designed to learn from the data and make predictions or decisions based on that data.\n3. **Training**: In machine learning, the algorithm is trained on a dataset to learn the patterns and relationships within the data.\n4. **Model**: Once the algorithm is trained, it creates a model that can be used to make predictions or decisions on 

In [26]:
simple_prompt = ChatPromptTemplate.from_template(
"""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:"""
)

simple_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])

In [27]:
retriver = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)
retriver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D5D3100ED0>, search_kwargs={'k': 3})

In [28]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [29]:
simple_rag_chain = (
    {"context":retriver | format_docs ,"question":RunnablePassthrough()}
    |simple_prompt
    |llm
    |StrOutputParser()
)

In [30]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D5D3100ED0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object a

In [32]:
res = simple_rag_chain.invoke("What is the NLP?")
res

'Based on the provided context, Natural Language Processing (NLP) is a branch of AI that helps computers understand human language. It combines computational linguistics with machine learning and deep learning.'